# 02 — Score TMDB Movies with NRC VAD + Emotion Intensity

Loads the full TMDB dataset and the pre-built keyword lexicon
(`output/tmdb_keyword_lexicon.csv` from `04_keyword_lexicon.ipynb`),
then aggregates keyword scores to movie level and exports to CSV.

**Prerequisites:** run `04_keyword_lexicon.ipynb` first to build the keyword lexicon.


In [1]:
import os
import time
import zipfile
from pathlib import Path

import kagglehub
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DATASET  = 'asaniczka/tmdb-movies-dataset-2023-930k-movies'
NROWS    = None   # None = full dataset; set to 10_000 for quick tests

EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD_DIMS = ['valence', 'arousal', 'dominance']


In [2]:
# ── Load TMDB dataset ─────────────────────────────────────────────────────
t0 = time.time()
base = Path(kagglehub.dataset_download(DATASET))
csvs = list(base.rglob('*.csv')) or []
if not csvs:
    zips = sorted(base.rglob('*.zip'))
    import zipfile
    with zipfile.ZipFile(zips[0]) as z:
        csvs = [z.extract(n, base) for n in z.namelist() if n.endswith('.csv')]

pandas_kwargs = {'nrows': NROWS} if NROWS else {}
df = pd.read_csv(sorted(csvs)[0], **pandas_kwargs)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

Loaded 1,383,070 rows in 5.7s


In [3]:
# ── Load pre-built keyword lexicon ───────────────────────────────────────
# Built by 04_keyword_lexicon.ipynb: one row per unique TMDB keyword,
# scored against all three NRC lexicons (EmoLex, Intensity, VAD v2.1).
LEXICON_CSV = Path('output/tmdb_keyword_lexicon.csv')
kw_df = pd.read_csv(LEXICON_CSV)
print(f'Keyword lexicon: {len(kw_df):,} keywords')

# Build kw_scores dict: keyword -> {emotion: float, vad_dim: float}
# Matches the interface expected by score_movie() and classify_keywords()
kw_scores = {}
for _, row in kw_df.iterrows():
    scores = {}
    for e in EMOTIONS:
        val = row[f'intensity_{e}']
        scores[e] = float(val) if pd.notna(val) else 0.0
    for d in VAD_DIMS:
        val = row[f'vad_{d}']
        scores[d] = float(val) if pd.notna(val) else float('nan')
    kw_scores[row['keyword']] = scores

coverage = kw_df[['emolex_found','intensity_found','vad_found']].mean() * 100
print(f'Coverage — EmoLex: {coverage["emolex_found"]:.1f}%  '
      f'Intensity: {coverage["intensity_found"]:.1f}%  '
      f'VAD: {coverage["vad_found"]:.1f}%')


Keyword lexicon: 67,916 keywords


Coverage — EmoLex: 55.5%  Intensity: 31.5%  VAD: 65.7%


In [4]:
# ── Verify keyword coverage against this TMDB slice ─────────────────────
tmdb_keywords = {
    kw.strip()
    for s in df['keywords'].dropna().astype(str)
    for kw in s.split(',') if kw.strip()
}
in_lexicon = tmdb_keywords & set(kw_scores.keys())
print(f'Unique keywords in this TMDB slice: {len(tmdb_keywords):,}')
print(f'Found in lexicon: {len(in_lexicon):,} ({len(in_lexicon)/len(tmdb_keywords):.1%})')
missing = len(tmdb_keywords) - len(in_lexicon)
print(f'Missing (will score as 0/NaN): {missing:,} — re-run 04_keyword_lexicon.ipynb to update')


Unique keywords in this TMDB slice: 67,939
Found in lexicon: 67,901 (99.9%)
Missing (will score as 0/NaN): 38 — re-run 04_keyword_lexicon.ipynb to update


In [5]:
# ── Aggregate to movie level (optimised) ─────────────────────────────────
t0 = time.time()

def score_movie(keywords_str):
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {**{e: 0.0 for e in EMOTIONS}, **{d: float('nan') for d in VAD_DIMS}}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]

    e_rows = [kw_scores[kw] for kw in kws if kw in kw_scores]
    emotion_means = {}
    for e in EMOTIONS:
        vals = [r[e] for r in e_rows if r[e] > 0]
        emotion_means[e] = sum(vals) / len(vals) if vals else 0.0

    vad_means = {}
    for d in VAD_DIMS:
        vals = [r[d] for r in e_rows if r[d] == r[d]]  # NaN-safe
        vad_means[d] = sum(vals) / len(vals) if vals else float('nan')

    return {**emotion_means, **vad_means}


# pd.DataFrame(tolist()) is ~10x faster than .apply(pd.Series) on large frames
scores_df = pd.DataFrame(df['keywords'].map(score_movie).tolist())
scores_df['sentiment'] = scores_df['valence'].apply(
    lambda v: 'positive' if v > 0 else ('negative' if v < 0 else 'neutral')
    if v == v else 'unknown'
)

elapsed = time.time() - t0
print(f'Scored {len(df):,} movies in {elapsed:.1f}s ({len(df)/elapsed:,.0f} movies/sec)')
print()
print('Sentiment distribution:')
print(scores_df['sentiment'].value_counts().to_string())

Scored 1,383,070 movies in 3.4s (410,177 movies/sec)

Sentiment distribution:
sentiment
unknown     1057327
positive     214318
negative     109296
neutral        2129


In [6]:
# ── Classify keywords by valence ────────────────────────────────────────
# Buckets each movie's keywords by their individual NRC VAD valence score.
# positive: valence > 0.5  |  negative: valence < 0.5
# neutral:  valence == 0.5 |  unknown: no VAD coverage

def classify_keywords(keywords_str):
    if not isinstance(keywords_str, str) or not keywords_str.strip():
        return {'positive_keywords': '', 'negative_keywords': '', 'neutral_keywords': '', 'unknown_keywords': ''}
    kws = [kw.strip() for kw in keywords_str.split(',') if kw.strip()]
    pos, neg, unk = [], [], []
    for kw in kws:
        v = kw_scores.get(kw, {}).get('valence', float('nan'))
        if v != v:      unk.append(kw)   # NaN = no VAD coverage
        elif v >= 0:    pos.append(kw)
        else:           neg.append(kw)
    return {
        'positive_keywords': ', '.join(pos),
        'negative_keywords': ', '.join(neg),
        'unknown_keywords':  ', '.join(unk),
    }

kw_groups = pd.DataFrame(df['keywords'].map(classify_keywords).tolist())
print('Keyword group sample:')
sample = df[['title', 'keywords']].join(kw_groups)
display(sample[sample['positive_keywords'] != ''].head(3)[['title', 'positive_keywords', 'negative_keywords', 'unknown_keywords']])


Keyword group sample:


,title,positive_keywords,negative_keywords,unknown_keywords
0,Inception,"rescue, mission, dream, airplane, virtual real...","kidnapping, spy, manipulation, car crash, heist","paris, france, los angeles, california"
1,Interstellar,"rescue, future, spacecraft, artificial intelli...","race against time, dystopia, wormhole, famine,...",2060s
2,The Dark Knight,"joker, secret identity, superhero, anti hero, ...","sadism, chaos, crime fighter, scarecrow, vigil...",


In [7]:
import tracemalloc
import gc

# ── Performance benchmark ─────────────────────────────────────────────────
print('=' * 55)
print('PERFORMANCE BENCHMARK')
print('=' * 55)

# Movie scoring (with memory tracking)
gc.collect()
tracemalloc.start()
t_mv_start = time.perf_counter()
scores_list = df['keywords'].map(score_movie).tolist()
t_mv = time.perf_counter() - t_mv_start
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'Lexicon load:      {len(kw_scores):>8,} keywords  (from tmdb_keyword_lexicon.csv)')
print(f'Movie scoring:     {len(df):>8,} movies    {t_mv:.2f}s  ({len(df)/t_mv:,.0f} movies/sec)')
print(f'Peak memory:       {peak_bytes/1e6:>8.1f} MB')

# Coverage stats
has_valence  = scores_df['valence'].notna().sum()
has_emotion  = (scores_df[EMOTIONS].sum(axis=1) > 0).sum()
no_keywords  = df['keywords'].isna().sum()
print()
print('COVERAGE')
print('-' * 55)
print(f'Movies total:      {len(df):>8,}')
print(f'No keywords:       {no_keywords:>8,}  ({100*no_keywords/len(df):.1f}%)')
print(f'VAD valence match: {has_valence:>8,}  ({100*has_valence/len(df):.1f}%)')
print(f'Emotion signal:    {has_emotion:>8,}  ({100*has_emotion/len(df):.1f}%)')
print(f'Keyword lexicon:   {len(kw_scores):>8,}  unique keywords (pre-scored)')
print('=' * 55)
del scores_list


PERFORMANCE BENCHMARK


Lexicon load:        67,916 keywords  (from tmdb_keyword_lexicon.csv)
Movie scoring:     1,383,070 movies    24.30s  (56,913 movies/sec)
Peak memory:          827.8 MB

COVERAGE
-------------------------------------------------------
Movies total:      1,383,070
No keywords:       1,037,756  (75.0%)
VAD valence match:  325,743  (23.6%)
Emotion signal:     230,349  (16.7%)
Keyword lexicon:     67,916  unique keywords (pre-scored)


In [8]:
# ── Sanity check ─────────────────────────────────────────────────────────
display_cols = ['title', 'keywords', 'valence', 'arousal', 'dominance', 'sentiment']
sanity_df = df[['title', 'keywords']].join(scores_df[['valence', 'arousal', 'dominance', 'sentiment']])
print('Most positive valence:')
display(sanity_df.nlargest(5, 'valence')[display_cols])
print()
print('Most negative valence:')
display(sanity_df.nsmallest(5, 'valence')[display_cols])

Most positive valence:


,title,keywords,valence,arousal,dominance,sentiment
85230,Black Bart,"california, wells fargo",1.0,-0.259,0.148,positive
87017,The Eye Above the Well,well,1.0,-0.259,0.148,positive
98537,Patisserie Coin De Rue,patisserie,1.0,-0.667,0.000,positive
101039,Tiara Tahiti,tahiti,1.0,-0.296,-0.296,positive
175664,Paradise,tahiti,1.0,-0.296,-0.296,positive



Most negative valence:


,title,keywords,valence,arousal,dominance,sentiment
4371,The Whole Ten Yards,hitman,-1.0,0.917,1.000,negative
6769,Proud Mary,hitman,-1.0,0.917,1.000,negative
12620,Survive Style 5+,hitman,-1.0,0.917,1.000,negative
15015,Carancho,car accident,-1.0,0.880,-0.448,negative
20610,Mr. Long,hitman,-1.0,0.917,1.000,negative


In [9]:
import json

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'movie_vad_scores.csv'

# Column order: identity → emotion scores → remaining TMDB metadata
# This ensures the Kaggle preview shows the most meaningful columns first
IDENTITY   = ['id', 'title', 'keywords']
KW_GROUPS  = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
SCORED     = ['sentiment', 'valence', 'arousal', 'dominance'] + EMOTIONS
TMDB_REST  = [c for c in df.columns if c not in IDENTITY + ['keywords']]

out_df = pd.concat([df[IDENTITY], kw_groups[KW_GROUPS], scores_df[SCORED], df[TMDB_REST]], axis=1)
out_df.to_csv(OUTPUT_CSV, index=False)

size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f'Wrote {len(out_df):,} rows x {len(out_df.columns)} cols -> {OUTPUT_CSV}  ({size_mb:.1f} MB)')
print(f'Column order: {list(out_df.columns[:8])} ... {list(out_df.columns[-4:])}')

# Write dataset metadata
print('Metadata ready at output/dataset-metadata.json')

Wrote 1,383,070 rows x 39 cols -> output/movie_vad_scores.csv  (642.4 MB)
Column order: ['id', 'title', 'keywords', 'positive_keywords', 'negative_keywords', 'unknown_keywords', 'sentiment', 'valence'] ... ['genres', 'production_companies', 'production_countries', 'spoken_languages']
Metadata ready at output/dataset-metadata.json
